# Converse2D v4: SRAM-resident fused training on A100

Choose **Runtime > Change runtime type > A100** first. `tools/v4_flash_fft/flash_study.py`:
1. builds the checked production extension and the research `flash_fft` extension;
2. for every case, asks the capacity switch whether the plane fits this GPU's opt-in shared memory per block
   (A100: 163 KB, so s1 100x100, s2 128x128 and s3 144x144 all run fused);
3. gates output and every input gradient against production FP32 and an FP64 reference, checks two calls bit for bit,
   and times forward+VJP against production with paired CUDA events.

`seed_sweep.py` repeats the accuracy gate over 8 seeds for the s2/s3 shapes. Expect about 20-30 minutes, mostly the builds.


In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

In [ ]:
%cd /content
!rm -rf ConverseNet
!git clone --branch claude/v4-dev https://github.com/Yiozolm/ConverseNet.git
%cd /content/ConverseNet
!git log --oneline -3
!pip -q install ninja

In [ ]:
import os, time
os.environ.setdefault('MAX_JOBS', str(os.cpu_count() or 2))
OUT = f"/content/results/flash-{torch.cuda.get_device_name(0).replace(' ', '_')}-{time.strftime('%Y%m%dT%H%M%S')}"
!python -u tools/v4_flash_fft/flash_study.py --output {OUT} 2>&1 | tee /content/flash_study.log

In [ ]:
!python -u tools/v4_flash_fft/seed_sweep.py --output {OUT}/seed_sweep.json --cases forward_s2_b4_c64_64 forward_s2_b4_c64_32 forward_s3_b2_c32_48 forward_s3_b2_c32_24 2>&1 | grep -E 'seed|eligible|Error' | tee {OUT}/seed_sweep.log

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT}/summary.md').read()))

Download the results and send back the whole zip (summary.md, flash.json, seed_sweep.json and the logs).

In [ ]:
import shutil
shutil.copy('/content/flash_study.log', OUT)
archive = shutil.make_archive(OUT, 'zip', OUT)
from google.colab import files
files.download(archive)